# 10 — Build v02 model configuration

Apply v01 lessons to a new model configuration:

1. **Datum offset**: apply the mean bias to `waterlevelbnd_constant_*.bc` to correct model low-bias
2. **Replacement obs points for BS and AE**: move to always-wet cells to avoid drying artifacts
3. **Add 3 central obs points (C1, C2, C3)** for better 3D vertical structure analysis

Approach: **copy the v01 directory to a new `dflowfm_v02/` folder** and modify only the needed files. The original v01 stays intact for comparison.

## 1. Imports and paths

In [ ]:
%matplotlib inline
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu

project_root = Path(r'F:\StagnoneDT')
v01_dir = project_root / 'model' / 'dflowfm'
v02_dir = project_root / 'model' / 'dflowfm_v02'

his_file = v01_dir / 'output' / 'Stagnone_dxy01_15m_his.nc'
map_file = v01_dir / 'output' / 'Stagnone_dxy01_15m_map.nc'
processed_dir = project_root / 'data' / 'processed'

## 2. Copy v01 model directory to v02

Copies all input files but **excludes** the `output/` folder (22.8 GB) and cache file.

In [ ]:
# Files/dirs to exclude when copying
EXCLUDE = {'output', '__pycache__'}
EXCLUDE_SUFFIX = {'.cache'}

def copy_v01_to_v02(src, dst):
    os.makedirs(dst, exist_ok=True)
    n_copied = 0
    for item in src.iterdir():
        if item.name in EXCLUDE or item.suffix in EXCLUDE_SUFFIX:
            print(f'  skip: {item.name}')
            continue
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
        n_copied += 1
    return n_copied

if v02_dir.exists():
    print(f'v02 directory already exists: {v02_dir}')
    print('  (files will be overwritten in the next cells)')
else:
    n = copy_v01_to_v02(v01_dir, v02_dir)
    print(f'Copied {n} items from v01 to v02')

# Also make an empty output dir
os.makedirs(v02_dir / 'output', exist_ok=True)

## 3. Compute datum offset from v01 bias

Load v01 results, compute mean bias per station, and take the average as the single offset to apply at the boundary (since the boundary BC affects the whole model globally).

In [ ]:
# Load v01 his file
ds_his = xr.open_dataset(str(his_file))
station_names = [s.decode().strip() if isinstance(s, bytes) else str(s).strip()
                  for s in ds_his.station_name.values]

def find_station(target):
    for i, n in enumerate(station_names):
        if target.lower() in n.lower():
            return i
    return None

idx = {
    'BocaNord': find_station('BocaNord'),
    'BocaSud': find_station('BocaSud'),
    'AltaVilaEst': find_station('AltaVilaEst') or find_station('Altavila'),
}

obs_files = {
    'BocaNord': 'wl_BocaNord_10min_UTC.csv',
    'BocaSud': 'wl_BocaSud_10min_UTC.csv',
    'AltaVilaEst': 'wl_AltavilaEst_10min_UTC.csv',
}

# Compute per-station bias
biases = {}
for name, i in idx.items():
    if i is None:
        continue
    model_wl = ds_his.waterlevel.isel(station=i).to_pandas()
    obs = pd.read_csv(str(processed_dir / obs_files[name]), index_col=0, parse_dates=True)
    obs.columns = ['wl_obs']
    df = pd.concat([model_wl.rename('wl_model'), obs['wl_obs']], axis=1).dropna()
    df = df[df.index >= df.index[0] + pd.Timedelta(hours=12)]
    biases[name] = (df['wl_model'] - df['wl_obs']).mean()

print('Per-station bias (model - obs):')
for name, b in biases.items():
    print(f'  {name:14s}: {b:+.4f} m')

avg_bias = np.mean(list(biases.values()))
print(f'\nAverage bias: {avg_bias:+.4f} m')
print(f'To correct: subtract {avg_bias:+.4f} from model WL')
print(f'Equivalent: set constant WL boundary offset to {-avg_bias:+.4f} m')

## 4. Update `waterlevelbnd_constant_*.bc` with the offset

The file has a single constant water level value (currently 0.0). We set it to `-avg_bias` so the boundary is raised/lowered to match observations.

In [ ]:
bc_file = v02_dir / 'waterlevelbnd_constant_Stagnone_dxy01_15m.bc'

# Read original
with open(bc_file, 'r') as f:
    original = f.read()
print('--- Original .bc file ---')
print(original)

# The last non-empty line is the constant value — replace it
offset_value = -avg_bias  # flip sign: we want to lift model WL

new_content = (
    '[General]\n'
    'fileVersion           = 1.01                \n'
    'fileType              = boundConds          \n'
    '\n'
    '[Forcing]\n'
    'name                  = Stagnone_dxy01_15m_bnd1_0001\n'
    'function              = constant            \n'
    'quantity              = waterlevelbnd       \n'
    'unit                  = m\n'
    f'{offset_value:.4f}\n'
)

with open(bc_file, 'w') as f:
    f.write(new_content)

print(f'\n--- Updated .bc file (offset = {offset_value:+.4f} m) ---')
with open(bc_file, 'r') as f:
    print(f.read())

## 5. Find replacement cells for BocaSud and AltaVilaEst

BocaNord was OK in v01 (deeper channel, no drying issues). BS and AE need to be moved to always-wet cells.

In [ ]:
# Open map file and load min depth per face
uds = xu.open_dataset(str(map_file), chunks={'time': 50})
face_coords = uds.grid.face_coordinates
face_x = np.asarray(face_coords[:, 0])
face_y = np.asarray(face_coords[:, 1])
bl = uds['mesh2d_flowelem_bl'].values

print('Computing min waterdepth per face (~30s)...')
min_depth_per_face = uds['mesh2d_waterdepth'].min(dim='time').compute().values
print(f'Done. Always-wet cells (min depth > 0.15 m): {(min_depth_per_face > 0.15).sum()} / {len(min_depth_per_face)}')

In [ ]:
MIN_DEPTH_THRESHOLD = 0.15
SEARCH_RADIUS_DEG = 0.005  # ~500m

# Original station coordinates from his.nc
station_coords = {}
for name, i in idx.items():
    if i is None:
        continue
    x = float(ds_his.station_x_coordinate.isel(station=i))
    y = float(ds_his.station_y_coordinate.isel(station=i))
    station_coords[name] = (x, y)

# Find always-wet replacement cell for each station (not needed for BN but checked for completeness)
replacement_coords = {}
print(f'{"Station":<14} {"orig":<22} {"new":<22} {"dist_m":>8} {"minD":>8} {"bed":>8}')
print('-' * 85)
for name, (x0, y0) in station_coords.items():
    dist2 = (face_x - x0) ** 2 + (face_y - y0) ** 2
    candidates = np.where((dist2 < SEARCH_RADIUS_DEG ** 2) &
                           (min_depth_per_face > MIN_DEPTH_THRESHOLD))[0]
    if len(candidates) == 0:
        print(f'{name}: no always-wet cell within {SEARCH_RADIUS_DEG*111000:.0f}m')
        replacement_coords[name] = (x0, y0)
        continue
    deepest = candidates[np.argmin(bl[candidates])]
    nx, ny = float(face_x[deepest]), float(face_y[deepest])
    d_m = np.sqrt(dist2[deepest]) * 111000 * np.cos(np.radians(y0))
    replacement_coords[name] = (nx, ny)
    orig = f'({x0:.4f},{y0:.4f})'
    new = f'({nx:.4f},{ny:.4f})'
    print(f'{name:<14} {orig:<22} {new:<22} {d_m:>8.1f} {min_depth_per_face[deepest]:>8.3f} {bl[deepest]:>8.3f}')

## 6. Build new observation file with replacement + central points

In [ ]:
obs_file = v02_dir / 'Stagnone_dxy01_15m_obs.xyn'

# Central points for 3D analysis (adjust based on preferred deeper locations)
central_points = [
    ('C1_Central', 12.455, 37.870),
    ('C2_NorthCenter', 12.460, 37.890),  # your adjusted C2
    ('C3_SouthCenter', 12.458, 37.855),
]

# Read current obs file
obs_df = pd.read_csv(obs_file, sep=r'\s+', header=None, names=['x', 'y', 'name'])
print(f'Original obs file: {len(obs_df)} stations\n')
print(obs_df.to_string(index=False))

# Update coordinates of BS and AE to replacement cells (keep BN as-is)
name_map = {'BocaSud': 'BocaSud', 'AltaVilaEst': 'AltaVilaEst', 'AltavilaEst': 'AltaVilaEst'}
updates = {'BocaSud', 'AltaVilaEst'}

for i, row in obs_df.iterrows():
    # Normalize the name
    base_name = row['name']
    for orig, key in [('BocaSud', 'BocaSud'), ('AltaVilaEst', 'AltaVilaEst'),
                      ('Altavila', 'AltaVilaEst')]:
        if orig.lower() in base_name.lower():
            if key in updates and key in replacement_coords:
                nx, ny = replacement_coords[key]
                obs_df.at[i, 'x'] = nx
                obs_df.at[i, 'y'] = ny
                print(f'\nUpdated {base_name}: -> ({nx:.4f}, {ny:.4f})')
            break

# Append central points
new_rows = pd.DataFrame(central_points, columns=['name', 'x', 'y'])[['x', 'y', 'name']]
obs_df = pd.concat([obs_df, new_rows], ignore_index=True)
print(f'\nAdded central points: {len(central_points)}')
print(f'Total stations in v02: {len(obs_df)}')

In [ ]:
# Write the new obs file in the expected xyn format (x y name, space-separated)
with open(obs_file, 'w') as f:
    for _, row in obs_df.iterrows():
        f.write(f'{row["x"]:>25.6f} {row["y"]:>25.6f} {row["name"]}\n')

print(f'Saved: {obs_file}')
print(f'\nFinal obs file:\n')
with open(obs_file, 'r') as f:
    print(f.read())

## 7. Verify v02 directory contents

In [ ]:
print(f'=== {v02_dir} ===')
for f in sorted(v02_dir.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:60s} {size_kb:>10.1f} KB')
    else:
        print(f'  {f.name}/')

print(f'\nKey changes vs v01:')
print(f'  - waterlevelbnd_constant_*.bc: offset = {offset_value:+.4f} m (was 0.0000)')
for name, (x0, y0) in station_coords.items():
    if name in replacement_coords:
        nx, ny = replacement_coords[name]
        if (nx, ny) != (x0, y0):
            print(f'  - {name} obs point moved: ({x0:.4f},{y0:.4f}) -> ({nx:.4f},{ny:.4f})')
print(f'  - added {len(central_points)} central obs points: {[c[0] for c in central_points]}')

## 8. Update run script for v02

In [ ]:
# The run_model.bat should already be in v02_dir (copied from v01).
# It calls dimr.exe with dimr_config.xml which references Stagnone_dxy01_15m.mdu.
# Since we kept the same filenames, nothing needs to change.
# Just verify the bat file is there:
bat = v02_dir / 'run_model.bat'
print(f'Run script: {bat} (exists: {bat.exists()})')
print(f'\nTo run v02:')
print(f'  cd {v02_dir}')
print(f'  run_model.bat')
print(f'\nExpected runtime: ~2h 45min (same mesh/layers/period as v01)')

## 9. Summary

**v02 changes vs v01:**

| Item | v01 | v02 |
|------|-----|-----|
| WL boundary offset | 0.000 m | computed from bias |
| BocaSud obs | original | moved to always-wet cell within ~500m |
| AltaVilaEst obs | original | moved to always-wet cell within ~500m |
| Central 3D points | none | C1, C2, C3 added |
| Everything else | unchanged | unchanged |

**Next steps:**
1. Run v02: `cd model/dflowfm_v02 && run_model.bat`
2. Post-process v02 results (re-use notebook 20_valid_v01_wl/09 with path changed to `dflowfm_v02/output/`)
3. Compare v02 vs v01 metrics — should see improvements in BocaSud/AltaVilaEst amplitude fit
4. If v02 passes, iterate on remaining issues: wind blending, mesh refinement